# Part 5 — Work IQ in a Microsoft Agent Framework agent

Now we let an **LLM drive** Work IQ: an agent that plans, reads the user's work context, and (with
approval) acts — all through Work IQ as an MCP tool. This is the same code that ships as the hosted
**`workmate-agent`** in `src/workmate-agent/` and publishes to Teams.

> An AI agent uses an LLM to run tools in a loop to achieve a goal.

## 1. Point an Agent Framework agent at Work IQ (MCP)

In [1]:
import os, sys, httpx
from agent_framework import Agent, MCPStreamableHTTPTool, tool
from agent_framework.azure import AzureAIAgentClient
from azure.identity import AzureCliCredential
from datetime import date
from dotenv import load_dotenv
load_dotenv("../.env")
sys.path.append(os.path.abspath(".."))
from notebooks._shared import get_user_token, WORK_IQ_GATEWAY

# The agent's LLM runs in Foundry, but the Work IQ MCP tool must call as the SIGNED-IN USER.
# We acquire the user's delegated Work IQ token and attach it to a custom httpx client that the
# MCP tool reuses. Note the generous timeout: Work IQ `ask` can take 10-30s to ground an answer.
token = get_user_token()

@tool
def get_current_date() -> str:
    """Return today's date in ISO format."""
    return date.today().isoformat()

work_iq_http = httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}", "Accept": "application/json, text/event-stream"},
    timeout=httpx.Timeout(180.0),
)
work_iq_tool = MCPStreamableHTTPTool(
    name="work-iq",
    url=os.environ.get("WORK_IQ_MCP_ENDPOINT", f"{WORK_IQ_GATEWAY}/mcp"),
    allowed_tools=["ask", "fetch", "search_paths", "get_schema", "do_action"],
    load_prompts=False,
    http_client=work_iq_http,
    request_timeout=180,
)
print("Work IQ token acquired:", token[:16], "...")
print("MCP tool configured, allowed tools:", work_iq_tool.allowed_tools)

Work IQ token acquired: eyJ0eXAiOiJKV1Qi ...
MCP tool configured, allowed tools: ['ask', 'fetch', 'search_paths', 'get_schema', 'do_action']


## 2. Give it a goal and instructions

In [2]:
# AzureAIAgentClient runs the LLM turn in the shared Foundry project (reuses gpt-5.4-mini).
client = AzureAIAgentClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model_deployment_name=os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-5.4-mini"),
    credential=AzureCliCredential(),
)

agent = Agent(
    client=client,
    name="ContosoWorkmate",
    instructions=(
        "You are Contoso's workmate assistant. For any question about the signed-in user's "
        "email, meetings, files, or people, call the Work IQ `ask` tool with a natural-language "
        "question - it grounds the answer in their Microsoft 365 context. Use fetch/search_paths/"
        "get_schema only for structured reads, and do_action to take action (always show a draft "
        "for approval before sending)."
    ),
    tools=[get_current_date, work_iq_tool],
)
print(f"Agent '{agent.name}' ready on model",
      os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-5.4-mini"),
      "with tools: get_current_date, work-iq")

Agent 'ContosoWorkmate' ready on model gpt-5.4-mini with tools: get_current_date, work-iq


## 3. Run it

> Work IQ needs **user context**. Locally use a signed-in credential; in production the Teams digital worker flows the user's token on-behalf-of (see `docs/teams.md`).

In [3]:
# The MCP tool opens its streamable-HTTP session inside the context manager; the Foundry client
# is cleaned up by its own context manager.
async with client, work_iq_tool:
    result = await agent.run(
        "What action items do I currently have based on my recent emails and Teams messages? "
        "List them by priority and note who is waiting on me."
    )
print(result.text if hasattr(result, "text") else result)

Here are your current action items from recent emails and Teams-related notifications, prioritized:

1. **Respond to Maria Garcia’s refund request**  
   - **Priority:** High  
   - **What to do:** Process or reply to the customer’s request for a **full refund** on a delayed order.  
   - **Who is waiting on you:** **Maria Garcia**

2. **Investigate the Seattle stock shortage and coordinate inventory transfer**  
   - **Priority:** High  
   - **What to do:** Check inventory for the **Professional Claw Hammer (SKU HTHM001600)**, see whether stock can be moved to Seattle, and escalate to procurement if there’s a wider shortage.  
   - **Who is waiting on you:** **Marcus Chen**, **Priya Sharma**, and **Jordan Lee**

3. **Review Global Administrator assignments made outside PIM**  
   - **Priority:** Medium  
   - **What to do:** Review the reported privileged role assignments and decide whether corrective action is needed.  
   - **Who is waiting on you:** Not explicitly identified

4. *

### Recap & next steps

- Work IQ plugs into Agent Framework as a single MCP tool with read + `do_action`.
- The hosted `workmate-agent` is this agent, deployable with `azd deploy workmate-agent`.
- Publish to Teams for real user-context OBO — `docs/teams.md`.

**That's the Work IQ deep dive.** Foundry IQ (Day 1) grounds in a knowledge base; Work IQ (Day 2)
grounds in *your* work — and acts. Fabric IQ (Day 3) grounds in business entities.